# Reasoning Models: o1, o3, DeepSeek-R1, and the Future of Test-Time Compute

## What Makes Reasoning Models Different?

Standard LLMs (GPT-4, Claude 3.5) generate tokens one after another — each token is predicted based on the previous context. They are trained to be helpful and produce answers directly. **Reasoning models** are different in a fundamental way: they are trained to **think before they answer**.

### Key Reasoning Models (2024–2025)

| Model | Developer | Release | Strength |
|-------|-----------|---------|----------|
| **o1** | OpenAI | Sep 2024 | Math, coding, science (STEM) |
| **o3** | OpenAI | Dec 2024 | Frontier reasoning, ARC-AGI |
| **o3-mini** | OpenAI | Jan 2025 | Fast, efficient reasoning |
| **DeepSeek-R1** | DeepSeek | Jan 2025 | Open-source, matches o1 |
| **DeepSeek-R1-Zero** | DeepSeek | Jan 2025 | Emergent reasoning via RL only |
| **Gemini 2.0 Flash Thinking** | Google | Dec 2024 | Fast reasoning, multimodal |
| **Claude 3.7 Sonnet** | Anthropic | Feb 2025 | Extended thinking mode |

### The Core Difference: Internal Chain of Thought

Reasoning models generate a **hidden scratchpad** (called the **reasoning trace** or **thinking tokens**) before producing their final answer:

```
Standard LLM:
  User Prompt → [Model] → Answer

Reasoning Model:
  User Prompt → [Model] → <thinking>
                              Step 1: Let me analyze...
                              Step 2: I should check...
                              Step 3: The answer must be...
                          </thinking> → Final Answer
```

This "thinking" is produced by reinforcement learning — the model was trained with rewards for **getting correct answers**, and it learned on its own to use intermediate reasoning steps as a strategy.

### How They Are Trained

- **o1/o3:** OpenAI uses reinforcement learning with verifiable rewards (math answers, code execution results). The exact training details are proprietary.
- **DeepSeek-R1:** Trained in two phases:
  1. **Cold Start SFT:** Fine-tune on human-written chain-of-thought data
  2. **Reasoning RL (GRPO):** Reinforce using rule-based reward signals (correctness, format)
  3. **Rejection Sampling + SFT:** Distill the best reasoning traces back into supervised fine-tuning
  4. **Final RL:** Polish with combined rewards

> **Key insight:** Reasoning ability is not just about bigger models — it is about spending more compute **at inference time** to search for better answers.

In [ ]:
# Cell 2: Setup and API Calls to Reasoning Models

# Install required packages
# pip install openai anthropic

import os
import time
from typing import Optional

# ─────────────────────────────────────────────
# 2a. OpenAI o1 / o3 API
# ─────────────────────────────────────────────
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', 'your-key-here'))

def call_reasoning_model(
    prompt: str,
    model: str = 'o3-mini',
    reasoning_effort: str = 'medium',  # 'low', 'medium', 'high'
    max_completion_tokens: int = 8000,
) -> dict:
    """
    Call an OpenAI reasoning model (o1, o3, o3-mini).
    
    reasoning_effort controls how long the model thinks:
      - 'low':    fast, less thorough (~few hundred thinking tokens)
      - 'medium': balanced (default)
      - 'high':   most thorough, slowest, highest cost
    """
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=max_completion_tokens,
        reasoning_effort=reasoning_effort,  # o3-mini and o3 support this
    )

    elapsed = time.time() - start
    usage   = response.usage

    return {
        'answer':                 response.choices[0].message.content,
        'model':                  response.model,
        'input_tokens':           usage.prompt_tokens,
        'output_tokens':          usage.completion_tokens,
        'reasoning_tokens':       getattr(usage.completion_tokens_details, 'reasoning_tokens', 'N/A'),
        'elapsed_seconds':        round(elapsed, 2),
        'reasoning_effort':       reasoning_effort,
    }


# Example usage (requires a valid API key)
math_problem = """
A ball is thrown vertically upward with an initial velocity of 20 m/s.
Taking g = 10 m/s², find:
1. The maximum height reached
2. The time to reach maximum height
3. The total time of flight
Show all working step by step.
"""

# Uncomment to run (requires API key)
# result = call_reasoning_model(math_problem, model='o3-mini', reasoning_effort='medium')
# print(f"Model: {result['model']}")
# print(f"Reasoning tokens used: {result['reasoning_tokens']}")
# print(f"Total output tokens: {result['output_tokens']}")
# print(f"Time taken: {result['elapsed_seconds']}s")
# print(f"\nAnswer:\n{result['answer']}")

print("OpenAI reasoning model function defined.")
print()

# ─────────────────────────────────────────────
# 2b. Anthropic Claude Extended Thinking
# ─────────────────────────────────────────────
import anthropic

def call_claude_extended_thinking(
    prompt: str,
    model: str = 'claude-3-7-sonnet-20250219',
    budget_tokens: int = 5000,   # max thinking tokens
    max_tokens: int = 8000,
) -> dict:
    """
    Call Claude with extended thinking enabled.
    budget_tokens: how many tokens Claude can use for internal reasoning.
    """
    anthropic_client = anthropic.Anthropic(
        api_key=os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')
    )

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        thinking={
            "type": "enabled",
            "budget_tokens": budget_tokens
        },
        messages=[{"role": "user", "content": prompt}]
    )

    thinking_text = ""
    answer_text   = ""

    for block in response.content:
        if block.type == 'thinking':
            thinking_text = block.thinking
        elif block.type == 'text':
            answer_text = block.text

    return {
        'thinking':         thinking_text,
        'answer':           answer_text,
        'input_tokens':     response.usage.input_tokens,
        'output_tokens':    response.usage.output_tokens,
    }


# Example (requires API key)
# result = call_claude_extended_thinking(math_problem, budget_tokens=3000)
# print("--- THINKING TRACE ---")
# print(result['thinking'][:500], "...")  # first 500 chars of thinking
# print("\n--- FINAL ANSWER ---")
# print(result['answer'])

print("Anthropic extended thinking function defined.")
print()

# ─────────────────────────────────────────────
# 2c. DeepSeek-R1 via OpenAI-compatible API
# ─────────────────────────────────────────────
def call_deepseek_r1(
    prompt: str,
    max_tokens: int = 4000,
) -> dict:
    """
    Call DeepSeek-R1 via DeepSeek's OpenAI-compatible API.
    The model returns a reasoning_content field separately.
    """
    deepseek_client = OpenAI(
        api_key=os.environ.get('DEEPSEEK_API_KEY', 'your-key-here'),
        base_url='https://api.deepseek.com'
    )

    response = deepseek_client.chat.completions.create(
        model='deepseek-reasoner',
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )

    msg = response.choices[0].message
    return {
        'reasoning':     getattr(msg, 'reasoning_content', ''),  # CoT trace
        'answer':        msg.content,
        'input_tokens':  response.usage.prompt_tokens,
        'output_tokens': response.usage.completion_tokens,
    }


print("DeepSeek-R1 function defined.")
print()
print("All reasoning model API functions ready.")

## Chain-of-Thought Prompting Strategies

Chain-of-thought (CoT) prompting is a technique to elicit step-by-step reasoning from LLMs. It was introduced in [Wei et al. 2022](https://arxiv.org/abs/2201.11903) and rapidly became a standard technique.

### Why CoT Works

When a model reasons step-by-step:
1. It decomposes complex problems into simpler sub-problems
2. Each intermediate step becomes context for the next step
3. The model is less likely to "jump" to a wrong answer
4. Errors are more visible and correctable

### Types of Chain-of-Thought Prompting

| Type | Trigger | Best For | Example |
|------|---------|----------|----------|
| **Zero-shot CoT** | "Think step by step" | Any problem | Add magic phrase to any prompt |
| **Few-shot CoT** | Exemplars with reasoning | Structured tasks | Provide 3–5 solved examples |
| **Self-consistency** | Multiple CoT paths, vote | Ambiguous problems | Sample 10–20 responses, take majority |
| **Tree of Thoughts (ToT)** | Explicit tree search | Planning, search | Branch and evaluate multiple paths |
| **ReAct** | Reasoning + Acting | Agent tasks | Interleave thinking with tool calls |
| **Least-to-Most** | Decompose then solve | Complex problems | Solve subproblems in order |

### Zero-Shot CoT

The simplest form — just add a phrase that triggers reasoning:

```python
# Without CoT
prompt = "A shop sells apples for $0.50 each and pears for $0.75 each. \
           John buys 4 apples and 3 pears. How much does he spend?"

# With Zero-shot CoT
prompt = """A shop sells apples for $0.50 each and pears for $0.75 each.
John buys 4 apples and 3 pears. How much does he spend?

Let's think step by step."""
```

**Effective zero-shot CoT trigger phrases:**
- "Let's think step by step."
- "Let's work through this carefully."
- "Think through this step by step before giving the final answer."
- "First, let me understand the problem. Then I'll solve it step by step."

### Few-Shot CoT

Provide examples that demonstrate the reasoning format you want:

```
Q: Roger has 5 tennis balls. He buys 2 more cans of 3 tennis balls each.
   How many tennis balls does he have now?

A: Roger starts with 5 tennis balls.
   He buys 2 cans × 3 balls = 6 balls.
   Total = 5 + 6 = 11 tennis balls.
   The answer is 11.

Q: [Your actual question here]
A:
```

### Self-Consistency CoT

Sample multiple independent chains of thought, then take the most common answer — this is more reliable than a single sample:

```
Sample 1: ... → Answer: 42
Sample 2: ... → Answer: 42
Sample 3: ... → Answer: 45   (minority)
Sample 4: ... → Answer: 42
                                → Final: 42 (majority vote)
```

In [ ]:
# Cell 4: Implementing Chain-of-Thought Prompting

import os
from openai import OpenAI
from collections import Counter
import re

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', 'demo'))

def zero_shot_cot(
    problem: str,
    model: str = 'gpt-4o',
    temperature: float = 0.0,
) -> str:
    """Zero-shot chain-of-thought: just append the magic phrase."""
    prompt = f"{problem}\n\nLet's think step by step."

    # Simulated response for demonstration
    print(f"[Zero-shot CoT Prompt]\n{prompt}\n")
    return "<API response with step-by-step reasoning>"


def few_shot_cot(
    problem: str,
    exemplars: list[dict],
    model: str = 'gpt-4o',
    temperature: float = 0.0,
) -> str:
    """
    Few-shot chain-of-thought with worked examples.
    
    exemplars: list of {'question': str, 'reasoning': str, 'answer': str}
    """
    # Build prompt from exemplars
    examples_text = ""
    for i, ex in enumerate(exemplars, 1):
        examples_text += f"Example {i}:\n"
        examples_text += f"Q: {ex['question']}\n"
        examples_text += f"A: {ex['reasoning']}\nThe answer is {ex['answer']}.\n\n"

    full_prompt = f"{examples_text}Now solve:\nQ: {problem}\nA:"
    print(f"[Few-shot CoT Prompt (truncated)]\n...{full_prompt[-200:]}")
    return "<API response>"


def self_consistency_cot(
    problem: str,
    model: str = 'gpt-4o',
    n_samples: int = 10,
    temperature: float = 0.7,
) -> tuple[str, Counter]:
    """
    Self-consistency: sample N reasoning paths, return majority answer.
    Each sample independently solves the problem with CoT.
    """
    prompt = f"{problem}\n\nLet's think step by step. End your response with 'The answer is: [answer]'"
    answers = []

    # Simulate N samples (in practice, call API N times)
    simulated_responses = [
        "Step 1: ... Step 2: ... The answer is: 42",
        "First... Then... The answer is: 42",
        "Let me think... The answer is: 45",  # minority
        "Working through: ... The answer is: 42",
        "Step-by-step: ... The answer is: 42",
    ]

    for response_text in simulated_responses[:n_samples]:
        # Extract final answer
        match = re.search(r'The answer is:?\s*([^\n\.]+)', response_text, re.IGNORECASE)
        if match:
            answers.append(match.group(1).strip())

    vote_counts   = Counter(answers)
    majority_answer = vote_counts.most_common(1)[0][0] if vote_counts else "No answer"

    return majority_answer, vote_counts


# ─── Demonstration ───────────────────────────────────────────────────────────

problem = """In a class of 30 students, 18 play football and 15 play basketball.
5 students play neither sport. How many students play both?"""

print("=" * 60)
print("ZERO-SHOT CoT")
print("=" * 60)
zero_shot_cot(problem)

print("\n" + "=" * 60)
print("FEW-SHOT CoT")
print("=" * 60)
arithmetic_exemplars = [
    {
        'question': 'There are 23 birds. 5 fly away. 12 more arrive. How many birds?',
        'reasoning': 'Start with 23. After 5 fly away: 23 - 5 = 18. After 12 arrive: 18 + 12 = 30.',
        'answer': '30'
    },
    {
        'question': 'A bag has 60 red and blue marbles. 40% are red. How many are blue?',
        'reasoning': 'Red marbles: 60 × 0.4 = 24. Blue marbles: 60 - 24 = 36.',
        'answer': '36'
    }
]
few_shot_cot(problem, arithmetic_exemplars)

print("\n" + "=" * 60)
print("SELF-CONSISTENCY CoT")
print("=" * 60)
majority, votes = self_consistency_cot(problem, n_samples=5)
print(f"Vote distribution: {dict(votes)}")
print(f"Majority answer:   {majority}")
print()
print("Correct answer: 30 play football or basketball. 30 = 18 + 15 - both")
print("Students playing sport = 30 - 5 = 25. Both = 18 + 15 - 25 = 8")

## When to Use Reasoning Models vs. Standard LLMs

Reasoning models are **not** always better. They are slower and significantly more expensive. Knowing when to use them is a key engineering skill.

### Use Reasoning Models When:

| Scenario | Why Reasoning Helps | Example |
|----------|--------------------|---------|
| **Multi-step math** | Errors compound — careful steps prevent cascades | Algebra, calculus, physics |
| **Complex coding** | Algorithm design requires planning, not just syntax | System design, bug investigation |
| **Logical deduction** | Multiple constraints must all be satisfied | Logic puzzles, constraint satisfaction |
| **Scientific analysis** | Requires interpreting evidence, not just recalling facts | Research Q&A, paper review |
| **Planning under constraints** | Multiple objectives, dependencies | Project planning, resource allocation |
| **Code debugging** | Need to trace execution, eliminate hypotheses | Complex bugs, race conditions |

### Use Standard LLMs When:

| Scenario | Why Standard Is Better | Example |
|----------|----------------------|----------|
| **Simple Q&A** | Instant answer, no reasoning needed | "What is the capital of France?" |
| **Creative writing** | Creativity != reasoning | Story generation, brainstorming |
| **Summarization** | Pattern matching, not deduction | Document summarization |
| **Translation** | Learned pattern, no logical steps | Language translation |
| **High-volume, low-latency** | Reasoning models are 5–20x slower | Customer chatbot, autocomplete |
| **Conversational AI** | Users expect fast responses | General chat, FAQ bot |

### Decision Framework

```
Is the task well-defined with a verifiable correct answer?
          │
    YES ──┴── NO → Use standard LLM
          │
Does it require multiple sequential logical steps?
          │
    YES ──┴── NO → Use standard LLM (or zero-shot CoT)
          │
Is latency < 2 seconds required?
          │
    YES ──┴── NO → Use reasoning model
          │
Use standard LLM + few-shot CoT
```

### Cost Comparison (approximate, Jan 2025)

| Model | Input ($/1M tokens) | Output ($/1M tokens) | Reasoning tokens |
|-------|--------------------|--------------------|------------------|
| GPT-4o | $2.50 | $10 | None |
| o3-mini (medium) | $1.10 | $4.40 | ~2,000 hidden |
| o3-mini (high) | $1.10 | $4.40 | ~10,000 hidden |
| o1 | $15 | $60 | ~5,000–30,000 hidden |
| o3 | $10 | $40 | Variable |
| DeepSeek-R1 (API) | $0.55 | $2.19 | Shown to user |
| Claude 3.7 Sonnet | $3 | $15 | Budget configurable |

> **Practical tip:** For most production use cases, `o3-mini` with `reasoning_effort='low'` or `'medium'` offers the best cost-performance tradeoff.

In [ ]:
# Cell 6: Benchmarking Reasoning vs Standard Models on Logic Tasks

import json
import time
from dataclasses import dataclass, field

@dataclass
class BenchmarkResult:
    model: str
    problem_id: str
    problem: str
    correct_answer: str
    model_answer: str
    is_correct: bool
    latency_seconds: float
    tokens_used: int


# Benchmark problems: logic puzzles with deterministic answers
BENCHMARK_PROBLEMS = [
    {
        'id': 'syllogism_1',
        'problem': 'All roses are flowers. All flowers need sunlight. Does a rose need sunlight? Answer yes or no.',
        'answer': 'yes',
        'type': 'deductive_reasoning'
    },
    {
        'id': 'arithmetic_1',
        'problem': 'A train leaves Station A at 9:00 AM traveling at 60 mph. Another train leaves Station B (180 miles away) at 10:00 AM traveling toward Station A at 90 mph. At what time do they meet? Answer in HH:MM format.',
        'answer': '11:00',
        'type': 'arithmetic_reasoning'
    },
    {
        'id': 'constraint_1',
        'problem': 'Alice, Bob, and Carol each have a different pet (cat, dog, fish). Alice does not have the dog. Bob does not have the cat. Carol has the fish. What pet does Alice have?',
        'answer': 'cat',
        'type': 'constraint_satisfaction'
    },
    {
        'id': 'counting_1',
        'problem': 'How many letters "r" are in the word "strawberry"? Answer with just the number.',
        'answer': '3',
        'type': 'counting' 
    },
    {
        'id': 'sequence_1',
        'problem': 'What is the next number in this sequence: 1, 1, 2, 3, 5, 8, 13, __? Answer with just the number.',
        'answer': '21',
        'type': 'pattern_recognition'
    },
]


def simulate_model_response(
    model_type: str,
    problem: dict,
) -> BenchmarkResult:
    """
    Simulate model responses for demonstration.
    In practice, replace with actual API calls.
    """
    # Simulated correct rates and latencies based on known model performance
    model_profiles = {
        'gpt-4o':     {'accuracy': {  # ~correct for each type
                         'deductive_reasoning': 0.98, 'arithmetic_reasoning': 0.75,
                         'constraint_satisfaction': 0.90, 'counting': 0.60, 'pattern_recognition': 0.95},
                       'latency': 1.2, 'tokens': 150},
        'o3-mini':    {'accuracy': {
                         'deductive_reasoning': 0.99, 'arithmetic_reasoning': 0.96,
                         'constraint_satisfaction': 0.99, 'counting': 0.95, 'pattern_recognition': 0.99},
                       'latency': 5.5, 'tokens': 1500},
        'gpt-3.5-turbo': {'accuracy': {
                         'deductive_reasoning': 0.88, 'arithmetic_reasoning': 0.55,
                         'constraint_satisfaction': 0.70, 'counting': 0.40, 'pattern_recognition': 0.88},
                       'latency': 0.5, 'tokens': 80},
    }

    import random
    random.seed(hash(model_type + problem['id']))

    profile   = model_profiles[model_type]
    acc_rate  = profile['accuracy'][problem['type']]
    is_correct = random.random() < acc_rate

    return BenchmarkResult(
        model           = model_type,
        problem_id      = problem['id'],
        problem         = problem['problem'],
        correct_answer  = problem['answer'],
        model_answer    = problem['answer'] if is_correct else 'wrong',
        is_correct      = is_correct,
        latency_seconds = profile['latency'] + random.uniform(-0.2, 0.2),
        tokens_used     = profile['tokens'] + random.randint(-20, 20),
    )


# Run benchmark
models_to_test = ['gpt-3.5-turbo', 'gpt-4o', 'o3-mini']
all_results: list[BenchmarkResult] = []

for model in models_to_test:
    for problem in BENCHMARK_PROBLEMS:
        result = simulate_model_response(model, problem)
        all_results.append(result)

# Summarize results
import pandas as pd

df = pd.DataFrame([
    {
        'model':       r.model,
        'type':        next(p['type'] for p in BENCHMARK_PROBLEMS if p['id'] == r.problem_id),
        'is_correct':  r.is_correct,
        'latency':     r.latency_seconds,
        'tokens':      r.tokens_used,
    }
    for r in all_results
])

summary = df.groupby('model').agg(
    accuracy    = ('is_correct', 'mean'),
    avg_latency = ('latency',   'mean'),
    avg_tokens  = ('tokens',    'mean'),
).round(3)

print("=" * 55)
print("BENCHMARK RESULTS: Reasoning vs Standard Models")
print("=" * 55)
print(summary.to_string())

print("\n--- Per Problem Type ---")
type_summary = df.pivot_table(
    values='is_correct', index='type', columns='model', aggfunc='mean'
).round(2)
print(type_summary.to_string())

## Cost and Latency Tradeoffs: When Is Reasoning Worth It?

### The Fundamental Tradeoff

Reasoning models spend compute thinking before answering. This means:
- **Higher accuracy** on complex tasks
- **Higher latency** (typically 5–30 seconds vs. 1–3 seconds)
- **Higher cost** (often 5–20x more expensive than standard models)

### When the Tradeoff is Worth It

**High-stakes, low-frequency tasks:**
```
If you run 100 medical diagnosis queries per day, and:
  - Standard GPT-4o:  80% accuracy, $0.002/query  → $0.20/day, 20 wrong answers/day
  - o3 (high effort): 97% accuracy, $0.04/query   → $4.00/day, 3 wrong answers/day

The extra $3.80/day to prevent 17 wrong answers is almost certainly worth it.
```

**Not worth it for high-volume, low-stakes tasks:**
```
If you run 100,000 customer support queries per day:
  - Standard GPT-4o:  $200/day
  - o3 (high effort): $4,000/day
  
The accuracy gain for simple FAQs likely doesn't justify 20x the cost.
```

### Routing Strategy: Hybrid Architecture

The best production systems **route** queries to the appropriate model:

```
User Query
    │
    ▼
Complexity Classifier (lightweight, fast)
    │
    ├─ Simple (FAQ, factual)     → GPT-4o-mini  ($0.0001/query)
    ├─ Moderate (analysis)       → GPT-4o       ($0.002/query)
    └─ Complex (reasoning, math) → o3-mini      ($0.01/query)
```

### Test-Time Compute Scaling

OpenAI's research showed that **scaling inference compute** can achieve results previously only possible with larger models. The o3 model on ARC-AGI:
- At low compute: ~4% (similar to previous GPT-4)
- At high compute: **87.5%** — superhuman on this benchmark

This is the "test-time compute scaling" phenomenon: you can trade inference dollars for accuracy dynamically.

In [ ]:
# Cell 8: Building a Reasoning-Augmented Pipeline for Complex Tasks

from __future__ import annotations
import os
from enum import Enum
from dataclasses import dataclass
from typing import Callable


class QueryComplexity(Enum):
    SIMPLE   = "simple"    # FAQ, factual, greeting
    MODERATE = "moderate"  # Analysis, summarization, explanation
    COMPLEX  = "complex"   # Multi-step math, coding, planning, deduction


@dataclass
class RoutingDecision:
    complexity:        QueryComplexity
    model:             str
    reasoning_effort:  str | None
    estimated_cost:    float  # in USD
    justification:     str


class ReasoningAugmentedPipeline:
    """
    A production-ready pipeline that:
    1. Classifies query complexity
    2. Routes to appropriate model (standard vs reasoning)
    3. Applies appropriate prompting strategy
    4. Returns structured results with metadata
    """

    # Simple keyword/heuristic classifier (in production: use a fine-tuned classifier)
    SIMPLE_SIGNALS   = ['what is', 'who is', 'when did', 'define', 'hello', 'hi',
                        'translate', 'summarize this:', 'what does']
    COMPLEX_SIGNALS  = ['prove', 'derive', 'debug', 'implement', 'optimize',
                        'solve', 'calculate', 'design a system', 'given that',
                        'step by step', 'algorithm', 'complexity', 'worst case']

    # Cost per 1000 tokens (output)
    MODEL_COSTS = {
        'gpt-4o-mini':           0.0006,
        'gpt-4o':                0.010,
        'o3-mini-low':           0.0044,
        'o3-mini-medium':        0.0044,
        'o3-mini-high':          0.0044,
    }

    def classify_complexity(self, query: str) -> QueryComplexity:
        """Classify query complexity using keyword heuristics."""
        query_lower = query.lower()

        # Complex signals take priority
        if any(sig in query_lower for sig in self.COMPLEX_SIGNALS):
            return QueryComplexity.COMPLEX

        # Length-based heuristic: long queries tend to be more complex
        word_count = len(query.split())
        if word_count > 80:
            return QueryComplexity.COMPLEX

        if any(sig in query_lower for sig in self.SIMPLE_SIGNALS):
            return QueryComplexity.SIMPLE

        return QueryComplexity.MODERATE

    def route(self, query: str) -> RoutingDecision:
        """Decide which model to use based on query complexity."""
        complexity = self.classify_complexity(query)

        routing_map = {
            QueryComplexity.SIMPLE: RoutingDecision(
                complexity        = complexity,
                model             = 'gpt-4o-mini',
                reasoning_effort  = None,
                estimated_cost    = 0.0001,
                justification     = 'Simple factual query — fast, cheap model sufficient'
            ),
            QueryComplexity.MODERATE: RoutingDecision(
                complexity        = complexity,
                model             = 'gpt-4o',
                reasoning_effort  = None,
                estimated_cost    = 0.002,
                justification     = 'Moderate complexity — standard capable model'
            ),
            QueryComplexity.COMPLEX: RoutingDecision(
                complexity        = complexity,
                model             = 'o3-mini',
                reasoning_effort  = 'medium',
                estimated_cost    = 0.012,
                justification     = 'Complex multi-step task — reasoning model needed'
            ),
        }

        return routing_map[complexity]

    def process(self, query: str, dry_run: bool = True) -> dict:
        """Full pipeline: classify → route → (optionally) call model."""
        decision = self.route(query)

        result = {
            'query':              query[:80] + '...' if len(query) > 80 else query,
            'complexity':         decision.complexity.value,
            'routed_to':          decision.model,
            'reasoning_effort':   decision.reasoning_effort,
            'estimated_cost_usd': decision.estimated_cost,
            'justification':      decision.justification,
        }

        if not dry_run:
            # In production: call the appropriate model API here
            result['answer'] = f"[Would call {decision.model} with effort={decision.reasoning_effort}]"

        return result


# ─── Demo ────────────────────────────────────────────────────────────────────
pipeline = ReasoningAugmentedPipeline()

test_queries = [
    "What is the capital of France?",
    "Can you summarize the key points of this meeting transcript?",
    "Prove that the square root of 2 is irrational using a proof by contradiction, and then implement a function in Python to verify rational approximations.",
    "Design a system that can handle 1 million concurrent users for a real-time ML prediction service. Include database, caching, and model serving layers.",
    "Debug this code: for i in range(10): print(i) — why doesn't it print 10?",
]

print(f"{'Query':<45} {'Complexity':<12} {'Model':<15} {'Cost ($)':<10}")
print("-" * 85)

total_cost = 0.0
for query in test_queries:
    r = pipeline.process(query)
    short_query = r['query'][:42] + '...'
    print(f"{short_query:<45} {r['complexity']:<12} {r['routed_to']:<15} ${r['estimated_cost_usd']:.4f}")
    total_cost += r['estimated_cost_usd']

print("-" * 85)
print(f"{'Total estimated cost:':<72} ${total_cost:.4f}")
print()
print("[Note: Routing to expensive models only when needed saves ~70–90% of API costs]")

## The Future of Reasoning Models and Test-Time Compute Scaling

### The Big Picture

We are witnessing a shift in how AI systems improve:

**Old paradigm (2020–2023): Scale training compute**
```
More data + bigger model + more training FLOPs → smarter model
GPT-2 → GPT-3 → GPT-4 (each required ~10x more training compute)
```

**New paradigm (2024–): Scale inference compute**
```
Smaller trained model + more thinking time at inference → smarter outputs
o3 can match much larger models by spending more tokens on reasoning
```

This is analogous to a person: a brilliant person can solve a hard problem faster, but given enough time, a diligent person can also solve it through careful step-by-step work.

### Scaling Laws for Test-Time Compute

OpenAI's research found that performance on hard math benchmarks scales as a **power law** with test-time compute:

```
Performance ∝ (test-time compute)^α
```

This means doubling the inference compute predictably improves performance — similar to how doubling training compute works during pre-training.

### What to Expect Next

| Trend | Implication for Engineers |
|-------|---------------------------|
| **Reasoning models become default** | Build systems expecting longer inference latency (5–30s) |
| **Dynamic compute allocation** | Systems will dynamically allocate more thinking for harder queries |
| **Open-source reasoning models** | DeepSeek-R1 shows OSS can match proprietary; expect more | 
| **Specialized reasoning** | Domain-specific reasoning models (legal, medical, scientific) |
| **Compound AI Systems** | Reasoning model + tools + agents = most powerful pattern |
| **Inference cost optimization** | Speculative decoding + caching for reasoning chains |

### The Compound AI System Frontier

The state-of-the-art architecture in 2025 is not a single model — it is a **compound AI system**:

```
User Query
    ↓
Reasoning Model (plans the approach)
    ↓
Tools: [Code execution] [Web search] [Database] [Calculator]
    ↓
Reasoning Model (synthesizes results)
    ↓
Final Answer with verified steps
```

Systems like OpenAI's Deep Research, Google's AlphaCode 2, and SWE-agent use exactly this pattern.

### Key Papers to Read

- **[Chain-of-Thought Prompting Elicits Reasoning in LLMs](https://arxiv.org/abs/2201.11903)** — Wei et al. 2022
- **[Self-Consistency Improves CoT Reasoning](https://arxiv.org/abs/2203.11171)** — Wang et al. 2022  
- **[Tree of Thoughts](https://arxiv.org/abs/2305.10601)** — Yao et al. 2023
- **[DeepSeek-R1 Technical Report](https://arxiv.org/abs/2501.12948)** — DeepSeek AI 2025
- **[Scaling LLM Test-Time Compute](https://arxiv.org/abs/2408.03314)** — Snell et al. 2024

In [ ]:
# Cell 10: Exercises — Practice with Reasoning Models

print("""
╔══════════════════════════════════════════════════════════════╗
║               EXERCISES — Reasoning Models                  ║
╚══════════════════════════════════════════════════════════════╝

Complete these 3 challenges to cement your understanding:
""")

# ─────────────────────────────────────────────────────────────────────────────
# Exercise 1: Implement a Complete Chain-of-Thought Benchmark
# ─────────────────────────────────────────────────────────────────────────────
print("EXERCISE 1: Build a Multi-Method CoT Benchmark")
print("-" * 55)
print("""
Implement a class `CoTBenchmark` that:

1. Accepts a list of math word problems with known answers
2. Tests each problem with THREE strategies:
   a) Direct answer (no CoT)
   b) Zero-shot CoT ("Let's think step by step")
   c) Self-consistency CoT (5 samples, majority vote)
3. Returns a comparison DataFrame showing:
   - Accuracy per strategy
   - Average response length per strategy
   - Estimated cost per strategy

Bonus: Add a 4th strategy using few-shot CoT with 2 exemplars.

Starter code:
""")

class CoTBenchmark:
    """YOUR IMPLEMENTATION HERE"""

    def __init__(self, problems: list[dict]):
        # problems: [{"question": str, "answer": str}]
        self.problems = problems

    def test_direct(self) -> list[dict]:
        """Test without any CoT."""
        raise NotImplementedError

    def test_zero_shot_cot(self) -> list[dict]:
        """Test with zero-shot CoT."""
        raise NotImplementedError

    def test_self_consistency(self, n_samples: int = 5) -> list[dict]:
        """Test with self-consistency (majority vote)."""
        raise NotImplementedError

    def run_all(self) -> 'pd.DataFrame':
        """Run all methods and return comparison DataFrame."""
        raise NotImplementedError


print("Exercise 1 template created. Implement the class above.")
print()

# ─────────────────────────────────────────────────────────────────────────────
# Exercise 2: Build a Smart Query Router
# ─────────────────────────────────────────────────────────────────────────────
print("EXERCISE 2: ML-Based Query Complexity Classifier")
print("-" * 55)
print("""
Improve the keyword-based router from Cell 8 with an ML classifier:

1. Create a labeled dataset of 50+ queries:
   - 20 simple (factual, conversational)
   - 15 moderate (analysis, explanation)
   - 15 complex (math, coding, multi-step)

2. Extract features from each query:
   - Word count, sentence count
   - Presence of question words (how, why, prove, calculate)
   - Contains numbers/formulas
   - Technical vocabulary ratio

3. Train a RandomForestClassifier on these features

4. Evaluate with 5-fold cross-validation

5. Compare with the keyword heuristic baseline from Cell 8

Goal: Achieve >85% accuracy on held-out test queries.
""")

# Starter dataset
query_dataset = [
    # (query, label) — 0=simple, 1=moderate, 2=complex
    ("What is Python?",                                                  0),
    ("Who invented the telephone?",                                       0),
    ("Translate 'hello' to Spanish.",                                    0),
    ("Summarize this article: [article]",                                 1),
    ("Explain how gradient descent works.",                               1),
    ("Prove that there are infinitely many prime numbers.",               2),
    ("Debug this recursive function and calculate its time complexity.",  2),
    ("Design a distributed system for 10M users with ML recommendations.", 2),
    # ADD 42+ MORE EXAMPLES HERE
]

print(f"Starter dataset has {len(query_dataset)} examples.")
print("Your task: expand to 50+ examples and train the classifier.")
print()

# ─────────────────────────────────────────────────────────────────────────────
# Exercise 3: Implement Tree of Thoughts
# ─────────────────────────────────────────────────────────────────────────────
print("EXERCISE 3: Implement Tree of Thoughts (ToT)")
print("-" * 55)
print("""
Tree of Thoughts (Yao et al. 2023) extends chain-of-thought by
exploring MULTIPLE reasoning paths and evaluating each:

                    Problem
                   /   |   \\
               Path1 Path2 Path3  (generate K thoughts)
                 |     |     |
              Eval  Eval  Eval   (score: sure/maybe/impossible)
                 |         |
              Best paths selected
                 |
              Continue...

Implement a simplified ToT for solving the "Game of 24":
Given 4 numbers, use +, -, *, / to make 24.

Example: [4, 7, 8, 8] → 4 * (7 - 8/8) = 4 * 6 = 24 ✓

Steps:
1. Generate all possible first operations on pairs of numbers
2. Evaluate if each intermediate result could lead to 24
3. Expand promising branches, prune impossible ones
4. Return a valid solution path or 'no solution'

Test on: [1, 1, 4, 6], [2, 3, 4, 8], [3, 4, 6, 8]
""")

from itertools import permutations
from fractions import Fraction

def solve_game_of_24_tot(numbers: list[int]) -> str | None:
    """
    Solve Game of 24 using Tree of Thoughts search.
    Returns a solution expression or None if impossible.
    
    YOUR IMPLEMENTATION HERE
    """
    raise NotImplementedError(
        "Implement this using recursive tree search."
        "Hint: at each step, pick two numbers, apply an operation, "
        "replace them with the result, recurse on the remaining numbers."
    )


# Test cases
test_cases = [
    ([4, 7, 8, 8], "Should find: 4*(7-8/8) = 24"),
    ([1, 1, 4, 6], "Should find: (4-1)*(6+1) is NOT 24... find the real solution"),
    ([2, 3, 4, 8], "Should find: 8/(3-2/4) = 8/(2.5) = NOT... find it!"),
]

print("Test cases for Game of 24:")
for nums, hint in test_cases:
    print(f"  {nums} — {hint}")

print()
print("""KEY LEARNING POINTS:
  1. Reasoning models = training to use internal CoT before answering
  2. Zero-shot CoT: add 'Think step by step' to any prompt
  3. Self-consistency: sample multiple paths, take majority vote
  4. Use reasoning models for: math, code, logic, planning
  5. Route queries: expensive reasoning only when needed
  6. Test-time compute scaling = future of AI improvement
""")